In [13]:
import cv2
import mediapipe as mp

In [14]:
model_path = './face_landmarker.task'

In [15]:
BaseOptions =  mp.tasks.BaseOptions
FaceLandmarker =  mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions =  mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode =  mp.tasks.vision.RunningMode

# Create a face landmarker instance with the video mode:
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1,
    output_face_blendshapes=True,
    min_face_presence_confidence=0.7)

In [16]:
# LEFT_IRIS  = 468
# RIGHT_IRIS = 473

In [17]:
def analyze_eye_contact(video_capture, frame_step ):
    results = []

    with FaceLandmarker.create_from_options(options) as landmarker:
        frame_index = 0
        fps = video_capture.get(cv2.CAP_PROP_FPS) or 30

        while True:
            _, frame = video_capture.read()
            if frame is None:
                break

            if frame_index % frame_step == 0:
                timestamp_ms = int((frame_index / fps) * 1000)
                mp_image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                )

                detection = landmarker.detect_for_video(mp_image, timestamp_ms)

                if detection.face_blendshapes:
                    # Extract eye blendshape values by name
                    blendshapes = {b.category_name: b.score for b in detection.face_blendshapes[0]}

                    eye_look_in    = (blendshapes.get('eyeLookInLeft',  0) + blendshapes.get('eyeLookInRight',  0)) / 2
                    eye_look_out   = (blendshapes.get('eyeLookOutLeft', 0) + blendshapes.get('eyeLookOutRight', 0)) / 2
                    eye_look_up    = (blendshapes.get('eyeLookUpLeft',  0) + blendshapes.get('eyeLookUpRight',  0)) / 2
                    eye_look_down  = (blendshapes.get('eyeLookDownLeft',0) + blendshapes.get('eyeLookDownRight',0)) / 2

                    # Average deviation from center (0 = looking at camera)
                    deviation = (eye_look_in + eye_look_out + eye_look_up + eye_look_down) / 4

                    # Convert to score: 1 = perfect eye contact, 0 = looking away
                    score = round(1 - deviation, 2)
                else:
                    score = 0

                results.append({"frame": frame_index, "eye_contact_score": score})

            frame_index += 1

    return results

In [20]:
cap = cv2.VideoCapture("C:\\Users\\beeko\\PycharmProjects\\AI-Video-interview-analysis-\\DataSet\\kaseb_video.mp4")  # 0 = webcam, or pass a file path
positions = analyze_eye_contact(cap, frame_step =10)
cap.release()
for p in positions:
    print(p)

{'frame': 0, 'eye_contact_score': 0.84}
{'frame': 10, 'eye_contact_score': 0.8}
{'frame': 20, 'eye_contact_score': 0.75}
{'frame': 30, 'eye_contact_score': 0.8}
{'frame': 40, 'eye_contact_score': 0.82}
{'frame': 50, 'eye_contact_score': 0.83}
{'frame': 60, 'eye_contact_score': 0}
{'frame': 70, 'eye_contact_score': 0.83}
{'frame': 80, 'eye_contact_score': 0.76}
{'frame': 90, 'eye_contact_score': 0}
{'frame': 100, 'eye_contact_score': 0.73}
{'frame': 110, 'eye_contact_score': 0.9}
{'frame': 120, 'eye_contact_score': 0.78}
{'frame': 130, 'eye_contact_score': 0.87}
{'frame': 140, 'eye_contact_score': 0.8}
{'frame': 150, 'eye_contact_score': 0.74}
{'frame': 160, 'eye_contact_score': 0.83}
{'frame': 170, 'eye_contact_score': 0.81}
{'frame': 180, 'eye_contact_score': 0.79}
{'frame': 190, 'eye_contact_score': 0.82}
{'frame': 200, 'eye_contact_score': 0.86}
{'frame': 210, 'eye_contact_score': 0.88}
{'frame': 220, 'eye_contact_score': 0.88}
{'frame': 230, 'eye_contact_score': 0}
{'frame': 240, '